# Sistema da Análise de Sentimentos

O sistema de análise de sentimentos foi feito usando como base, comentários para a avaliação de livros na Amazon, utilizando o dataset `Amazon Book Reviews 2023`. 

É possível enxergar este problema sob uma ótica de **aprendizado supervisionado**, onde a partir do *dataset* dos comentários das resenhas de livros da Amazon, iremos extrair algumas amostras e categorizar as análises como positivas, neutras ou negativas. Em seguida, um novo *dataset* com menos amostras e traduzidas será salvo, contendo além de uma coluna de comentários, uma coluna constando qual é a classe do comentário. 

Evidenciando um problema clássico de **classificação por multi-classe**, a tarefa do modelo pré-treinado exportado será processar o conjunto de treino que representa parte do *dataset* traduzido e devolver os valores crus das probabilidades, os quais serão passados por uma função de ativação *softmax* a qual será responsável por entregar um vetor contendo as probabilidades de cada uma das classes, dentre elas, a que tiver a maior probabilidade irá representar, consequentemente, a confiança do modelo em avaliar o sentimento daquele comentário. 

**OBS**: O intuito deste notebook é documentar todas as etapas realizadas para a criação do modelo de análise de sentimentos. Caso queira utilizar este notebook para teste, **não execute todas as células**. Vá para a sessão de `Teste do Modelo`, execute as últimas células e teste.

## Carregando o *dataset*

Como já mencionado anteriormente, o *dataset* que será aplicado será o `Amazon Book Reviews 2023`. Ele não foi utilizado de forma completa.

Nesse sentido, o conjunto de dados foi carregado no modo *streaming* para que ao invés de baixar tudo, baixar apenas uma quantidade escolhida de amostras (i.e. comentários), determinadas por `SAMPLE_SIZE`. Isso será necessário por que o tamanho original do conjunto de dados é de 750 GB.

In [1]:
# importando as biblitotecas para carregamento e manipulação do conjunto
from datasets import load_dataset
import pandas as pd

# carregando o dataset
raw_dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Books",
    split="full",
    streaming=True,
    trust_remote_code=True
)

SAMPLE_SIZE = 2000
samples= []

for item in raw_dataset:
    if len(samples) >= SAMPLE_SIZE:
        break
    text = item.get("text", "").strip()
    rating = item.get("rating", None)
    if text and rating is not None:
        samples.append({"text": text, "rating": float(rating)})

df = pd.DataFrame(samples)
print(f"Total de amostras coletadas para o dataframe criado: {len(df)}")
print(df['rating'].value_counts())
df.head()

/home/guilherme/Downloads/nn u3 atividade analise de sentimentos/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total de amostras coletadas para o dataframe criado: 2000
rating
5.0    1239
4.0     502
3.0     162
1.0      49
2.0      48
Name: count, dtype: int64


,text,rating
0,It is definitely not a watercolor book. The p...,1.0
1,Updated: after first book arrived very damaged...,5.0
2,I bought it for the bag on the front so it pai...,5.0
3,Updated: after 1st arrived damaged the replace...,5.0
4,I love this book! The patterns are lovely. I ...,5.0


## Mapeando as notas para as 3 classes

Haverão 3 classes para as notas:

- Negativo: 1 e 2 estrelas
- Neutro: 3 estrelas
- Positivo: 4 e 5 estrelas 

In [2]:
def mapear_sentimento(rating):
    """
    Função usada para mapear os comentários classificados em estrelas para bom, ruim ou neuutro
    """
    if (rating <= 2):
        return 0
    elif (rating == 3):
        return 1 
    else:
        return 2

# definindo uma coluna do dataset para as labels
df['label'] = df['rating'].apply(mapear_sentimento)
label_map = {0: 'Negativo', 1: 'Neutro', 2: 'Positivo'}

In [3]:
print(f"Distribuição das avaliações por classe")
print(df['label'].map(label_map).value_counts())

Distribuição das avaliações por classe
label
Positivo    1741
Neutro       162
Negativo      97
Name: count, dtype: int64


## Traduzindo os comentários

Essa etapa será responsável por traduzir os comentários em inglês extraídos do dataset e salvar o conjunto de dados traduzido.

In [12]:
from deep_translator import GoogleTranslator
import time 
# max_chars limita a quantidade a ser traduzida pela API
def traduzir_texto(texto, max_chars= 4500):
    """Função usada para traduiz o texto de inglês pra pt-br"""
    try:
        texto = texto[:max_chars]
        return GoogleTranslator(source='en', target='pt').translate(texto)
    except Exception as e:
        print(f'Erro na tradução: {e}')
        return texto # texto original

print("Processo de tradução inicializado")

textos_traduzidos = []
for i, texto in enumerate(df['text']):
    traduzido = traduzir_texto(texto)
    textos_traduzidos.append(traduzido)

    if (((i+1) % 100) == 0):
        print(f"    {i+1}/{len(df)} traduzidos...")
    
df['text_pt'] = textos_traduzidos
print("Tradulçao Concluída!")

Processo de tradução inicializado
    100/2000 traduzidos...
    200/2000 traduzidos...
    300/2000 traduzidos...
    400/2000 traduzidos...
    500/2000 traduzidos...
    600/2000 traduzidos...
    700/2000 traduzidos...
    800/2000 traduzidos...
    900/2000 traduzidos...
    1000/2000 traduzidos...
    1100/2000 traduzidos...
    1200/2000 traduzidos...
    1300/2000 traduzidos...
    1400/2000 traduzidos...
    1500/2000 traduzidos...
    1600/2000 traduzidos...
    1700/2000 traduzidos...
    1800/2000 traduzidos...
    1900/2000 traduzidos...
    2000/2000 traduzidos...
Tradulçao Concluída!


In [8]:
# Exemplo da tradução dos comentários
print("Original: \n", df['text'].iloc[0][:200])
print("\nTraduzido: \n", df['text_pt'].iloc[0][:200])

Original: 
 It is definitely not a watercolor book.  The paper bucked completely.  The pages honestly appear to be photo copies of other pictures. I say that bc if you look at the seal pics you can see the tell t


KeyError: 'text_pt'

In [4]:
# Salvando o dataset traduzido
df[['text_pt', 'label']].to_csv("dataset_livros_pt.csv", index=False)
print("Dataset salvo em: dataset_livros_pt.csv")

KeyError: "['text_pt'] not in index"

**OBS: Apenas Recarregue o Dataset**, caso já tenha salvo `dataset_livros_pt.csv`

In [5]:
df = pd.read_csv("dataset_livros_pt.csv")
print(f"Dataset recarregado: {len(df)} linhas")
print(df['label'].value_counts())

Dataset recarregado: 2000 linhas
label
2    1741
1     162
0      97
Name: count, dtype: int64


## Carregando o modelo + *tokenizer*

Agora iremos carregar o modelo `bert-base-portuguese-cased` juntamente com o tokenizador.

- `bert-base-portuguese-cased`

Trata-se de um modelo treinado em corpus brasileiro em partes (utilizando Wikipedia PT). Sendo "*cased*" pois ele é capaz de diferenciar letras maiúsculas das minúsculas. Falando de forma mais geral sobre a sua arquitetura, o modelo possui 12 camadas *Transformer*, com aproximadamente 110 milhões de parâmetros da rede. 

É daí que vem a necessidade de utilizar um dataset de comentários traduzidos em PT-BR, pois como o *bert-base-portuguese-cased* foi treinado em português, o conjunto de dados utilizado para treino precisa estar em português para que o *fine-tuning* faça sentido com as palavras em português que já foram treinadas durante a fase de pré-treino do modelo. 

- Tokenizador (*tokenizer*)

O tokenizador é a entidade responsável por realizar a etapa de tokenização do texto, quebrando as palavras e convertendo em vetores de ID's numéricos para que, quando inseridos no modelo *bert*, passe pelas etapas de *embedding* posicional e no bloco de *encoder*.

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_checkpoint= "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels= 3
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

#convertendo o dataset traduzido para o formato de HuggingFace Dataset
hf_dataset = Dataset.from_pandas(df[['text_pt', 'label']].rename(columns={'text_pt': 'text'}))
# função de tokenização 
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation= True,
        padding= "max_length",
        max_length= 128
    )

# aplicando a tokenização
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])


print(f"Dataset tokenizado: {len(tokenized_dataset)} exemplos")
print("Colunas:", tokenized_dataset.column_names)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 3726.79 examples/s]

Dataset tokenizado: 2000 exemplos
Colunas: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


## *Splitando* o conjunto em treino e validação

Com o conjunto de dados já tratado, o conjunto de dados será dividido em uma parte de treino e outra parte de validação. Ou seja, uma parte será utilizada para o aprendizado do modelo e a outra parte será utilizada como forma de avaliar o modelo.

In [8]:
split = tokenized_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

len(train_dataset), len(eval_dataset)

(1700, 300)

## Configurações de Treinamento

Aqui serão *setadas* as configurações de ajuste-fino (*fine-tuning*) do modelo.

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./resultados_livros",
    num_train_epochs= 3, # épocas pro fine-tuning
    per_device_train_batch_size= 8,
    per_device_eval_batch_size= 8,
    eval_strategy= "epoch", # avalia no final de cada época
    save_strategy= "epoch",
    logging_steps= 50,
    load_best_model_at_end= True,
    report_to="none"
)

## Configurações das Métricas de Avaliação

As métricas que estão sendo colocadas servirão como forma de medir o desempenho do modelo no conjunto de validação.

In [11]:
import numpy as np 
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions= np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

## Treinamento do Modelo

In [12]:
from transformers import Trainer

data_collator = DataCollatorWithPadding(tokenizer= tokenizer)

trainer = Trainer(
    model= model,
    args= training_args,
    train_dataset= train_dataset,
    eval_dataset= eval_dataset,
    data_collator= data_collator,
    compute_metrics= compute_metrics
)

print(f"Iniciando o fine-tuning do modelo...")
trainer.train()
print(f"\nTreinamento concluído!")

Iniciando o fine-tuning do modelo...


  8%|▊         | 50/639 [00:20<03:56,  2.49it/s]

{'loss': 0.578, 'grad_norm': 2.087819814682007, 'learning_rate': 4.6087636932707356e-05, 'epoch': 0.23}


 16%|█▌        | 100/639 [00:40<03:38,  2.46it/s]

{'loss': 0.4855, 'grad_norm': 3.2551121711730957, 'learning_rate': 4.217527386541471e-05, 'epoch': 0.47}


...

{'loss': 0.1831, 'grad_norm': 8.527379989624023, 'learning_rate': 3.051643192488263e-06, 'epoch': 2.82}


                                                 
100%|██████████| 639/639 [04:53<00:00,  2.61it/s]

{'eval_loss': 0.4269745945930481, 'eval_accuracy': 0.89, 'eval_f1': 0.8785666736009985, 'eval_runtime': 5.6198, 'eval_samples_per_second': 53.383, 'eval_steps_per_second': 6.762, 'epoch': 3.0}


100%|██████████| 639/639 [04:55<00:00,  2.16it/s]

{'train_runtime': 295.5946, 'train_samples_per_second': 17.253, 'train_steps_per_second': 2.162, 'train_loss': 0.2985700120761734, 'epoch': 3.0}

Treinamento concluído!


## Salvando o modelo treinado

In [13]:
model.save_pretrained("./modelo_sentimentos_livros")
tokenizer.save_pretrained("./modelo_sentimentos_livros")

print("Modelo salvo!")

Modelo salvo!


## Testando o modelo 

In [2]:
import torch

label_map = {0:'Negativo', 1:'Neutro', 2:'Positivo'}

def classificar_comentario(frase):
    inputs = tokenizer(
        frase,
        return_tensors= "pt",
        truncation= True, 
        padding= True, 
        max_length= 128
    )
    inputs = inputs.to(model.device)

    # fazendo a inferência -> gerando os logits do modelo
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # possibilidado = softmax(logits)
    probabilidades = torch.softmax(logits, dim= 1)[0]
    predicao = torch.argmax(logits, dim= 1).item()

    print(f"Frase : {frase}")
    print(f"Sentimento: {label_map[predicao]}")
    print(f"Confiança: {probabilidades[predicao].item()*100:.1f}%")
    print(f"    > Negativo: {probabilidades[0].item()*100:.1f}%")
    print(f"    > Neutro: {probabilidades[1].item()*100:.1f}%")
    print(f"    > Positivo: {probabilidades[2].item()*100:.1f}%\n")

frases_teste = [
    "Livro incrível, me emocionei bastante com a reta final dele!",
    "A história chega a ser interressante, mas o final decepciona bastante.",
    "Péssimo livro. O enredo é limitado e é muito mal escrito!",
    "É uma leitura razoável para passar o tempo.",
    "Uma das melhores obras que já li na minha vida!",
    "Achei ele bem mais ou menos. Não erra mas não agrada.",
    "Não fede nem cheira."
]

for frase in frases_teste:
    classificar_comentario(frase)

Frase : Livro incrível, me emocionei bastante com a reta final dele!
Sentimento: Positivo
Confiança: 99.7%
    > Negativo: 0.1%
    > Neutro: 0.2%
    > Positivo: 99.7%

Frase : A história chega a ser interressante, mas o final decepciona bastante.
Sentimento: Negativo
Confiança: 56.0%
    > Negativo: 56.0%
    > Neutro: 43.4%
    > Positivo: 0.6%

Frase : Péssimo livro. O enredo é limitado e é muito mal escrito!
Sentimento: Negativo
Confiança: 96.9%
    > Negativo: 96.9%
    > Neutro: 2.7%
    > Positivo: 0.5%

Frase : É uma leitura razoável para passar o tempo.
Sentimento: Positivo
Confiança: 94.1%
    > Negativo: 0.2%
    > Neutro: 5.7%
    > Positivo: 94.1%

Frase : Uma das melhores obras que já li na minha vida!
Sentimento: Positivo
Confiança: 99.7%
    > Negativo: 0.1%
    > Neutro: 0.2%
    > Positivo: 99.7%

Frase : Achei ele bem mais ou menos. Não erra mas não agrada.
Sentimento: Neutro
Confiança: 88.0%
    > Negativo: 9.9%
    > Neutro: 88.0%
    > Positivo: 2.1%

Frase : Não

## **OBS**: Carregando o modelo já salvo e testando

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("./modelo_sentimentos_livros")
model = AutoModelForSequenceClassification.from_pretrained("./modelo_sentimentos_livros")

classificar_comentario("Que livro maravilhoso, recomendo muito!")

Frase : Que livro maravilhoso, recomendo muito!
Sentimento: Positivo
Confiança: 99.7%
    > Negativo: 0.1%
    > Neutro: 0.2%
    > Positivo: 99.7%

